In [1]:
import pandas as pd
from pyproj import CRS, Transformer
import autoroot
from rs_tools._src.utils.io import get_list_filenames, get_list_filenames_af
from rs_tools._src.geoprocessing.match import match_timestamps_af
from pathlib import Path   
import numpy as np 
import xarray as xr
from satpy.scene import Scene
from pyhdf.SD import SD, SDC 
import pandas as pd
import cartopy.crs as ccrs
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import fnmatch
from pyproj import CRS, Transformer
from pyhdf.SD import SD, SDC 
import gc
from datetime import timedelta
import glob
%load_ext autoreload
%autoreload 2

/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/pyproj/__init__.py:95: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/goes2go/data.py:665: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  within=pd.to_timedelta(config["nearesttime"].get("within", "1h")),
/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/goes2go/NEW.py:188: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  within=pd.to_timedelta(config["nearesttime"].get("within", "1h")),


In [2]:
def convert_lat_lon_to_x_y(crs, lon, lat):
    transformer = Transformer.from_crs(CRS("+proj=latlon"), crs, always_xy=True)
    x, y = transformer.transform(lon, lat)
    return x, y

def parse_af_dates_from_file(file):
    timestamp = Path(file).name.split("_")[0]
    return timestamp

In [3]:
def create_fires_ds(matching_modis_files, modis_dir, msg):
    array = np.zeros((msg.y.size, msg.x.size))
    var = "msg_seviri_fes_3km"
    crs_wkt = msg[var].crs_wkt
    crs = CRS(crs_wkt)
    print(os.path.join(modis_dir, matching_modis_files[0]))
    f = SD(os.path.join(modis_dir, matching_modis_files[0]), SDC.READ)
    df = pd.DataFrame()
    sds_obj = f.select('fire mask') # select sds
    fm_data = sds_obj.get() # get sds data

    if fm_data[fm_data > 6].shape[0] > 0:
        for key in ['FP_latitude', 
                        'FP_longitude', 
                        'FP_power']:

            sds_obj = f.select(key) 
            
            data = sds_obj.get()
            df[key] = data.ravel()
    
        for index, row in df.iterrows():
            lon = row['FP_longitude']
            lat = row['FP_latitude']
            x_sel, y_sel = convert_lat_lon_to_x_y(crs, lon, lat)
            selected = msg.sel(x=x_sel, y=y_sel, method='nearest')
            # Get the indices of the nearest point
            x_idx = msg.get_index('x').get_loc(selected['x'].item())
            y_idx = msg.get_index('y').get_loc(selected['y'].item())
            array[y_idx, x_idx] = 1
        da = xr.DataArray(
            array,
            coords={"y": msg.y, "x": msg.x},
            dims=("y", "x")
    )
    else:
        print('No fires detected')
        return None
    return da

In [11]:
fb = pd.read_excel('/mnt/data8tb/fire_detection/fire_brigade/Dasikes_Pyrkagies_2023_v1.8.xlsx', skiprows=1)
msg_path = '/mnt/outputs/geoprocessed'
save_msg_path = '/mnt/data8tb/fire_detection/fire_brigade/patched_msg'
save_af_path = '/mnt/data8tb/fire_detection/fire_brigade/patched_af'
if not os.path.exists(save_msg_path):
    os.makedirs(save_msg_path)
if not os.path.exists(save_af_path):
    os.makedirs(save_af_path)

In [4]:
fb['sum'] = fb[['Δάση',
       'Δασική Έκταση', 'Άλση', 'Χορτ/κές Εκτάσεις', 'Καλάμια - Βάλτοι',
       'Γεωργικές Εκτάσεις', 'Υπολλείματα Καλλιεργειών', 'Σκουπι-δότοποι']].sum(axis=1)
fd = fb.sort_values('sum', ascending=False)
fd_sub = fd[fd['sum'] > 10]
fd_sub['timestamp'] = pd.to_datetime(fd_sub['Ημερ/νία Έναρξης'].astype(str) + ' ' + fd_sub['Ώρα Έναρξης'].astype(str), 
                                 format='%Y-%m-%d %H:%M')

#remove 3 hours from timestamp
fd_sub['timestamp_ut'] = fd_sub['timestamp'] - pd.Timedelta(hours=3)
fd_sub['timestamp_ut_str'] = fd_sub['timestamp_ut'].dt.strftime('%Y%m%d%H%M00')
unique_times_fb = fd_sub['timestamp_ut'].dt.strftime('%Y%m%d%H%M00').unique().tolist()

/tmp/ipykernel_1336564/2519916280.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fd_sub['timestamp'] = pd.to_datetime(fd_sub['Ημερ/νία Έναρξης'].astype(str) + ' ' + fd_sub['Ώρα Έναρξης'].astype(str),
/tmp/ipykernel_1336564/2519916280.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fd_sub['timestamp_ut'] = fd_sub['timestamp'] - pd.Timedelta(hours=3)
/tmp/ipykernel_1336564/2519916280.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

In [5]:
fd_sub

,Α/Α ΕΓΓΡΑΦΗΣ,Α/Α ENGAGE,X-ENGAGE,Y-ENGAGE,Υπηρεσία,Νομός,Ημερ/νία Έναρξης,Ώρα Έναρξης,Ημερ/νία Κατασβεσης,Ώρα Κατάσβεσης,...,Α/Φ CL415,Α/Φ CL215,Α/Φ PZL,Α/Φ GRU.,ΜΙΣΘ. ΕΛΙΚΟΠΤ.,ΜΙΣΘ. ΑΕΡΟΣΚ.,sum,timestamp,timestamp_ut,timestamp_ut_str
1274,1911773,1096267,26.175290,41.115503,Π.Κ. ΣΟΥΦΛΙΟΥ,ΕΒΡΟΥ,2023-08-21,13:05,2023-10-02,15:30,...,11,0,2,0,9.0,9.0,818348.00,2023-08-21 13:05:00,2023-08-21 10:05:00,20230821100500
6986,1907588,1084257,27.940565,36.232992,Π.Υ. ΡΟΔΟΥ,ΔΩΔΕΚΑΝΗΣΩΝ,2023-07-18,19:09,NaT,NaN,...,9,0,0,0,0.0,5.0,176398.00,2023-07-18 19:09:00,2023-07-18 16:09:00,20230718160900
980,1923198,1095402,26.078210,40.935618,Π.Υ. ΑΛΕΞΑΝΔΡΟΥΠΟΛΗΣ,ΕΒΡΟΥ,2023-08-19,04:40,2023-09-25,20:19,...,11,0,2,0,9.0,12.0,122144.00,2023-08-19 04:40:00,2023-08-19 01:40:00,20230819014000
6730,1891011,1083804,23.498074,38.189420,Π.Υ. ΟΙΝΟΦΥΤΩΝ,ΒΟΙΩΤΙΑΣ,2023-07-17,17:08,2023-08-14,09:08,...,9,7,2,0,0.0,0.0,110000.00,2023-07-17 17:08:00,2023-07-17 14:08:00,20230717140800
3385,1897313,1087558,22.749670,39.367687,Π.Υ. ΒΙ.ΠΕ.ΒΟΛΟΥ,ΜΑΓΝΗΣΙΑΣ,2023-07-26,13:35,2023-08-19,20:47,...,0,0,0,0,0.0,0.0,86340.00,2023-07-26 13:35:00,2023-07-26 10:35:00,20230726103500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2120,1929726,1135712,22.331521,40.538138,Π.Υ. ΒΕΡΟΙΑΣ,ΗΜΑΘΙΑΣ,2023-12-23,17:35,2023-12-23,19:00,...,0,0,0,0,0.0,0.0,10.10,2023-12-23 17:35:00,2023-12-23 14:35:00,20231223143500
681,1895397,1099471,24.153976,41.111824,Π.Υ. ΔΡΑΜΑΣ,ΔΡΑΜΑΣ,2023-08-29,14:22,2023-08-29,16:05,...,0,0,0,0,0.0,0.0,10.10,2023-08-29 14:22:00,2023-08-29 11:22:00,20230829112200
3456,1869916,1070265,21.810887,39.567101,Π.Υ. ΤΡΙΚΑΛΩΝ,ΤΡΙΚΑΛΩΝ,2023-06-05,15:49,2023-06-05,16:55,...,0,0,0,0,0.0,0.0,10.05,2023-06-05 15:49:00,2023-06-05 12:49:00,20230605124900
536,1850111,1048876,24.707864,40.954719,Π.Υ. ΧΡΥΣΟΥΠΟΛΗΣ,ΚΑΒΑΛΑΣ,2023-03-13,10:33,NaT,NaN,...,0,0,0,0,0.0,0.0,10.05,2023-03-13 10:33:00,2023-03-13 07:33:00,20230313073300


In [6]:
len(unique_times_fb)

862

In [7]:
#files_msg = get_list_filenames(msg_path, ".nc", "2023") #1min and 1.4sec

In [8]:
files_msg_af = get_list_filenames_af(msg_path, ".nc", "2023") #57.9sec

In [12]:
unique_times_msg = list(set(map(parse_af_dates_from_file, files_msg_af)))

In [13]:
len(unique_times_msg)

14124

In [14]:
df_matches = match_timestamps_af(unique_times_fb, unique_times_msg, cutoff=120)
df_matches.columns = ['timestamp_fb', 'timestamp_msg']

No valid af mask found for 2023-11-04 09:10:00
No valid af mask found for 2023-11-05 06:38:00
No valid af mask found for 2023-12-04 14:32:00
No matching af mask found for 2023-03-13 12:58:00
No valid af mask found for 2023-11-08 09:00:00
No valid af mask found for 2023-10-13 09:55:00
No matching af mask found for 2023-04-27 10:00:00
No valid af mask found for 2023-10-02 16:49:00
No matching af mask found for 2023-04-21 10:27:00
No valid af mask found for 2023-11-02 10:00:00
No matching af mask found for 2023-03-17 13:18:00
No matching af mask found for 2023-03-27 14:45:00
No matching af mask found for 2023-03-13 11:46:00
No valid af mask found for 2023-11-03 10:36:00
No matching af mask found for 2023-04-07 08:00:00
No matching af mask found for 2023-03-31 13:00:00
No valid af mask found for 2023-10-11 17:38:00
No valid af mask found for 2023-10-21 07:00:00
No matching af mask found for 2023-02-06 11:17:00
No matching af mask found for 2023-03-25 14:16:00
No valid af mask found for 202

In [15]:
df_matches

,timestamp_fb,timestamp_msg
0,20230821100500,20230821101241
1,20230718160900,20230718161242
2,20230819014000,20230819014242
3,20230717140800,20230717141241
4,20230726103500,20230726104241
...,...,...
497,20230816105800,20230816111241
498,20230804152100,20230804152741
499,20230806141200,20230806141241
500,20230829112200,20230829112742


In [14]:
timestamp_msg = '20230821095741'

In [16]:
def find_next_msg_files(timestamp_msg, steps=3):
    timestamp_msg_dt = datetime.strptime(timestamp_msg, "%Y%m%d%H%M%S")
    timestamps = [timestamp_msg_dt + timedelta(minutes=15 * i) for i in range(1, steps + 1)]
    timestamp_strings = [dt.strftime("%Y%m%d%H%M") for dt in timestamps]
    datasets = [os.path.join(msg_path, f"{timestamp_msg}_msg.nc")]
    for ts in timestamp_strings:
        file_pattern = os.path.join(msg_path, f"{ts}*_msg.nc")  # Match files without seconds
        files = glob.glob(file_pattern)
        if len(files) > 0:
            datasets.append(files[0])
    return datasets

In [18]:
def af_patch_from_msg(msg, y_idx, x_idx, random_n_x, random_n_y):
    array = np.zeros((msg.y.size, msg.x.size))
    array[y_idx, x_idx] = 1
    da = xr.DataArray(
        array,
        coords={"y": msg.y, "x": msg.x},
        dims=("y", "x")
    )
    da_sel = da.isel(x=slice(x_idx - random_n_x, x_idx + (32 - random_n_x)), y = slice(y_idx - random_n_y, y_idx + (32 - random_n_y)))
    return da_sel

In [19]:
def create_msg_af_patches(msg, crs, lat, lon, itime, step):
    x, y = convert_lat_lon_to_x_y(crs, lon, lat)
    selected = msg.sel(x=x, y=y, method='nearest')
    # Get the indices of the nearest point
    x_idx = msg.get_index('x').get_loc(selected['x'].item())
    y_idx = msg.get_index('y').get_loc(selected['y'].item())
    for i in range(1, 3):
        random_n_x = np.random.randint(4, 28)
        random_n_y = np.random.randint(4, 28)
        msg_sel = msg.isel(x=slice(x_idx - random_n_x, x_idx + (32 - random_n_x)), y = slice(y_idx - random_n_y, y_idx + (32 - random_n_y)))
        # concatenate variables
        msg_sel_temp = xr.concat(
            [msg_sel.cloud_mask, msg_sel.latitude, msg_sel.longitude], dim="band"
        )
        # name data variables "Rad"
        msg_sel_temp = msg_sel_temp.to_dataset(name="Rad")
        msg_sel_temp = msg_sel_temp.drop_vars(["cloud_mask", "latitude", "longitude"])
        msg_sel_temp = msg_sel_temp.assign_coords(
            band=["cloud_mask", "latitude", "longitude"]
        )
        # merge with original dataset
        ds = xr.merge([msg_sel_temp.Rad, msg_sel.Rad])
        # store band names to be attached to da later
        band_names = [str(i) for i in ds.band.values]
        del msg_sel_temp
        gc.collect()

        # extract radiance data array
        da = ds.Rad
        da.attrs["fb_datetime"] = str('test')
        da.attrs["msg_time"] = msg.time.values[0]
        da.attrs["step"] = step
        da_af = af_patch_from_msg(msg, y_idx, x_idx, random_n_x, random_n_y)
        file_path_msg = Path(save_msg_path).joinpath(
                        f"{itime}_patch_5000_v{i}_dt{step}.tif"
                )
                # remove file if it already exists
        if os.path.exists(file_path_msg):
            os.remove(file_path_msg)
        da.rio.to_raster(file_path_msg)
        file_path_af = Path(save_af_path).joinpath(
                        f"{itime}_patch_5000_v{i}_dt{step}.tif"
                )
                # remove file if it already exists
        if os.path.exists(file_path_af):
            os.remove(file_path_af)
        da_af.rio.to_raster(file_path_af)

In [22]:
for index, row in df_matches.iterrows():
    timestamp_fb = row['timestamp_fb']
    timestamp_msg = row['timestamp_msg']
    lat = fd_sub[fd_sub.timestamp_ut_str == timestamp_fb]['Y-ENGAGE'].values
    lon = fd_sub[fd_sub.timestamp_ut_str == timestamp_fb]['X-ENGAGE'].values
    images_msg = find_next_msg_files(timestamp_msg, steps = 3)
    for n_dt, tm_msg in enumerate(images_msg):
        print(f'Processing {n_dt}, {tm_msg}')
        msg = xr.open_dataset(os.path.join(msg_path, tm_msg))
        var = "msg_seviri_fes_3km"
        crs_wkt = msg[var].crs_wkt
        crs = CRS(crs_wkt)
        try:
            create_msg_af_patches(msg, crs, lat, lon, timestamp_msg , step = n_dt)
        except:
            print(f'Error in processing msg {timestamp_msg} and af {timestamp_fb}')

Processing 0, /mnt/outputs/geoprocessed/20230821101241_msg.nc
Processing 1, /mnt/outputs/geoprocessed/20230821102741_msg.nc
Processing 2, /mnt/outputs/geoprocessed/20230821104241_msg.nc
Processing 3, /mnt/outputs/geoprocessed/20230821105741_msg.nc
Processing 0, /mnt/outputs/geoprocessed/20230718161242_msg.nc
Processing 1, /mnt/outputs/geoprocessed/20230718162742_msg.nc
Processing 2, /mnt/outputs/geoprocessed/20230718164241_msg.nc
Processing 3, /mnt/outputs/geoprocessed/20230718165741_msg.nc
Processing 0, /mnt/outputs/geoprocessed/20230819014242_msg.nc
Processing 1, /mnt/outputs/geoprocessed/20230819015742_msg.nc
Processing 2, /mnt/outputs/geoprocessed/20230819021242_msg.nc
Processing 3, /mnt/outputs/geoprocessed/20230819022742_msg.nc
Processing 0, /mnt/outputs/geoprocessed/20230717141241_msg.nc
Processing 1, /mnt/outputs/geoprocessed/20230717142741_msg.nc
Processing 2, /mnt/outputs/geoprocessed/20230717144242_msg.nc
Processing 3, /mnt/outputs/geoprocessed/20230717145742_msg.nc
Processi

In [41]:
#read tif files with rioxarray
import rasterio

test = rioxarray.open_rasterio('/mnt/data8tb/fire_detection/fire_brigade/20230821101241_patch_5000_v2_st3.tif')

In [42]:
test

<xarray.DataArray (band: 14, y: 32, x: 32)> Size: 115kB
[14336 values with dtype=float64]
Coordinates:
  * band         (band) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * x            (x) float64 256B 1.95e+06 1.953e+06 ... 2.04e+06 2.043e+06
  * y            (y) float64 256B 3.886e+06 3.889e+06 ... 3.976e+06 3.979e+06
    spatial_ref  int64 8B 0
Attributes:
    fb_datetime:   test
    msg_time:      2023-08-21 10:45:00
    step:          3
    scale_factor:  1.0
    add_offset:    0.0
    long_name:     Rad